# Meeting Minutes Creator (Week 3) Project : [Google Colab Notebook Copy]

In [ ]:
# !pip install -U bitsandbytes>=0.46.1

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [ ]:
from google.colab import userdata
from huggingface_hub import login

key = userdata.get('HuggingFace')
login(key, add_to_git_credential=True)

In [ ]:
audio_file = "/content/denver_extract.mp3"

audio_file1 = open(audio_file, "rb")

# Step 1: Transcribe Audio:

## Option 1: Hugging Face Open Source Model:

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_file)
transcription = result["text"]
print(transcription)

In [ ]:
open_transcription = transcription

## Option 2: Using OpenAI model:

In [ ]:
from openai import OpenAI

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file1, response_format="text")
print(transcription)


In [ ]:
from IPython.display import Markdown, display

display(Markdown(transcription))

In [ ]:
display(Markdown(open_transcription))

# Step 2: Analyse the Text:

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=1000, streamer=streamer)